# 晶合集成 688249 — SuperMind 策略

**Step 1 完成检查：**
- 已在 SuperMind 研究环境打开本 Notebook
- 内核选择 **Python 3.8**

**下一步：** 依次运行下方单元格 → Step 2 回测

In [ ]:
SOURCE_CODE = r'''
# ===== 晶合集成 688249 — 趋势跟随 + 阶梯止盈 =====
STOCK = '688249.SH'

# 关键价位（与 jinghe-tracker/config.py 对齐）
R67 = 67.0
R61 = 61.0
S58 = 58.0
S52 = 52.0
VALUE_LO = 24.0
VALUE_HI = 30.0
TREND_START = 28.0

# 阈值
RSI_OB = 72
RSI_OS = 35
MA_DEV_WARN = 0.15
UPPER_SHADOW = 0.5

# 仓位
INIT_PCT = 0.45
ADD_PCT = 0.25
MAX_PCT = 0.70
LIGHT_SELL = 0.35
HEAVY_SELL = 0.55


def init(context):
    g.stock = STOCK
    context.security = STOCK
    g.last_signal = None
    log.info('晶合趋势策略初始化: {}'.format(g.stock))


def _rsi(closes, period=14):
    if len(closes) < period + 1:
        return 50.0
    gains, losses = [], []
    for i in range(1, len(closes)):
        d = closes[i] - closes[i - 1]
        gains.append(max(d, 0))
        losses.append(max(-d, 0))
    avg_g = sum(gains[-period:]) / period
    avg_l = sum(losses[-period:]) / period
    if avg_l == 0:
        return 100.0
    rs = avg_g / avg_l
    return 100 - 100 / (1 + rs)


def _macd_hist(closes):
    if len(closes) < 35:
        return 0.0, 0.0

    def ema(data, n):
        k = 2.0 / (n + 1)
        v = data[0]
        for x in data[1:]:
            v = x * k + v * (1 - k)
        return v

    ema12 = ema(closes[-35:], 12)
    ema26 = ema(closes[-35:], 26)
    dif = ema12 - ema26
    dea = dif * 0.8
    hist = (dif - dea) * 2
    prev_closes = closes[:-1]
    ema12p = ema(prev_closes[-35:], 12)
    ema26p = ema(prev_closes[-35:], 26)
    hist_prev = ((ema12p - ema26p) - (ema12p - ema26p) * 0.8) * 2
    return hist, hist_prev


def _analyze(context):
    df = history(
        g.stock,
        ['open', 'high', 'low', 'close', 'volume'],
        60,
        '1d',
        False,
        'pre',
        True,
    )
    if df is None or len(df) < 25:
        return None

    closes = list(df['close'])
    opens = list(df['open'])
    highs = list(df['high'])
    lows = list(df['low'])
    vols = list(df['volume'])

    c = closes[-1]
    h = highs[-1]
    l = lows[-1]
    o = opens[-1]
    prev_c = closes[-2]

    ma5 = sum(closes[-5:]) / 5
    ma10 = sum(closes[-10:]) / 10
    ma20 = sum(closes[-20:]) / 20
    rsi = _rsi(closes)
    hist, hist_prev = _macd_hist(closes)

    body = abs(c - o) if abs(c - o) > 0.01 else 0.01
    upper_ratio = (h - max(o, c)) / body
    dev_ma20 = (c - ma20) / ma20 if ma20 else 0
    vol_ma5 = sum(vols[-6:-1]) / 5 if len(vols) >= 6 else vols[-1]
    vol_ratio = vols[-1] / vol_ma5 if vol_ma5 else 1.0

    buy_score = 0
    sell_score = 0
    buy_reasons = []
    sell_reasons = []

    bull_align = ma5 > ma10 > ma20
    trend_up = c > ma20 and ma5 > ma20

    # --- 买入: 覆盖 26 元启动 → 60 元趋势 ---
    if VALUE_LO <= c <= VALUE_HI and rsi <= RSI_OS:
        buy_score += 3
        buy_reasons.append('价值区超卖')
    if trend_up and TREND_START <= c <= VALUE_HI:
        buy_score += 3
        buy_reasons.append('28-30趋势启动')
    if bull_align and c > ma20 and c <= VALUE_HI + 2:
        buy_score += 2
        buy_reasons.append('MA多头排列')
    if bull_align and c > ma10 and l <= ma10 * 1.02 and rsi <= 58:
        buy_score += 2
        buy_reasons.append('回踩MA10')
    if c > TREND_START and vol_ratio >= 1.15 and c > prev_c:
        buy_score += 1
        buy_reasons.append('放量上行')
    if hist > 0 and hist_prev <= 0:
        buy_score += 1
        buy_reasons.append('MACD转强')

    # --- 卖出: 60+ 高位阶梯止盈 ---
    if h >= R67 * 0.98 and c < R67:
        sell_score -= 3
        sell_reasons.append('67压力回落')
    if c >= R61 * 0.98 and upper_ratio >= UPPER_SHADOW:
        sell_score -= 2
        sell_reasons.append('61区上影')
    if rsi >= RSI_OB:
        sell_score -= 2
        sell_reasons.append('RSI超买{:.0f}'.format(rsi))
    if dev_ma20 >= MA_DEV_WARN:
        sell_score -= 1
        sell_reasons.append('偏离MA20')
    if upper_ratio >= UPPER_SHADOW and c >= S58:
        sell_score -= 1
        sell_reasons.append('长上影')
    if c < S58 * 0.995:
        sell_score -= 3
        sell_reasons.append('跌破58平台')
    if c < ma20 and prev_c >= ma20:
        sell_score -= 3
        sell_reasons.append('跌破MA20')
    if c < S52 * 0.995:
        sell_score -= 4
        sell_reasons.append('跌破52')
    if hist < 0 and hist_prev >= 0 and c >= R61 * 0.95:
        sell_score -= 1
        sell_reasons.append('高位MACD转弱')

    return {
        'close': c,
        'ma5': ma5,
        'ma10': ma10,
        'ma20': ma20,
        'rsi': rsi,
        'buy_score': buy_score,
        'sell_score': sell_score,
        'buy_reason': ';'.join(buy_reasons) if buy_reasons else '',
        'sell_reason': ';'.join(sell_reasons) if sell_reasons else '',
    }


def _position_pct(context, price):
    total = context.portfolio.total_value
    if total <= 0:
        return 0.0
    pos = context.portfolio.positions.get(g.stock)
    mv = pos.market_value if pos else 0.0
    return mv / total


def _buy(context, price, cash, pct, reason):
    budget = cash * pct
    amt = int(budget / price / 100) * 100
    if amt >= 100:
        order(g.stock, amt)
        g.last_signal = 'BUY:' + reason
        log.info('买入 {} 股 @ {:.2f}, {}'.format(amt, price, reason))


def _sell_ratio(context, hold, ratio, reason):
    """ratio: 0~1 卖出比例，按 100 股取整"""
    if ratio >= 0.99 and hold >= 100:
        order_target(g.stock, 0)
        g.last_signal = 'CLEAR:' + reason
        log.info('清仓, {}'.format(reason))
        return True
    amt = int(hold * ratio) // 100 * 100
    if amt >= 100:
        order(g.stock, -amt)
        g.last_signal = 'SELL:' + reason
        log.info('卖出 {} 股, {}'.format(amt, reason))
        return True
    return False


def handle_bar(context, bar_dict):
    a = _analyze(context)
    if not a:
        return

    pos = context.portfolio.positions.get(g.stock)
    hold = pos.amount if pos else 0
    price = bar_dict[g.stock].close
    cash = context.portfolio.cash
    pos_pct = _position_pct(context, price)

    log.info(
        '{} px={:.2f} buy={} sell={} hold={} pos={:.0%} b={} s={}'.format(
            get_datetime(),
            price,
            a['buy_score'],
            a['sell_score'],
            hold,
            pos_pct,
            a['buy_reason'] or '-',
            a['sell_reason'] or '-',
        )
    )

    # 1) 强制清仓
    if hold > 0 and a['sell_score'] <= -6:
        _sell_ratio(context, hold, 1.0, a['sell_reason'] or '多重止损')
        return

    # 2) 阶梯减仓（先卖后买）
    if hold > 0 and a['sell_score'] <= -4:
        _sell_ratio(context, hold, HEAVY_SELL, a['sell_reason'] or '重度减仓')
        return
    if hold > 0 and a['sell_score'] <= -2:
        _sell_ratio(context, hold, LIGHT_SELL, a['sell_reason'] or '轻度减仓')
        return

    # 3) 建仓 / 加仓
    if hold == 0 and a['buy_score'] >= 2:
        _buy(context, price, cash, INIT_PCT, a['buy_reason'] or '启动建仓')
        return

    if hold > 0 and pos_pct < MAX_PCT and a['buy_score'] >= 3 and a['sell_score'] > -2:
        _buy(context, price, cash, ADD_PCT, a['buy_reason'] or '趋势加仓')
'''

print('✓ 策略代码已加载，共', len(SOURCE_CODE), '字符')
print('✓ Step 1 就绪 — 请运行下一个单元格开始回测 (Step 2)')


In [ ]:
# Step 2: 回测（Step 1 完成后运行此格）
btest = research_strategy(
    SOURCE_CODE,
    start_date='20250301',
    end_date='20260703',
    capital_base=200000,
    frequency='DAILY',
    stock_market='STOCK',
    benchmark='000688.SH',
)

print('=== 持仓曲线 ===')
display(btest['analyser']['portfolio'])
print('=== 交易明细 ===')
display(btest['analyser']['trades'])